# 탐색적 데이터 분석 (EDA)
## AI 기반 고객 이탈 예측 및 리텐션 ROI 최적화 시스템

이 노트북은 시뮬레이터가 생성한 고객 행동 데이터를 탐색하고,
이탈률 15~25% 범위 검증 및 주요 패턴을 분석합니다.

**데이터 소스**: `python src/main.py --mode simulate` 실행 후 `data/raw/` 에 생성되는 CSV

| 파일 | 컬럼 |
|------|------|
| `customers.csv` | customer_id, persona, is_treatment, churned |
| `events.csv` | customer_id, event_date, event_type, persona, is_treatment, order_value |

## 1. 환경 설정 및 데이터 로드

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

# 한글 폰트 설정
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']:
    try:
        plt.rcParams['font.family'] = font_name
        break
    except Exception:
        continue
plt.rcParams['axes.unicode_minus'] = False

# 프로젝트 루트를 path에 추가 (src 모듈 import용)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print('라이브러리 로드 완료')

In [ ]:
# ── 데이터 로드 ──────────────────────────────────────────────
# 시뮬레이터 출력 스키마에 맞춰 직접 로드
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

customers = pd.read_csv(DATA_DIR / 'customers.csv')
events = pd.read_csv(DATA_DIR / 'events.csv')

# event_date → datetime 변환
events['event_date'] = pd.to_datetime(events['event_date'], errors='coerce')

# 파생: signup_date (고객별 첫 이벤트 날짜)
first_event = events.groupby('customer_id')['event_date'].min().rename('signup_date')
customers = customers.merge(first_event, on='customer_id', how='left')
customers['signup_date'] = customers['signup_date'].fillna(pd.Timestamp('2024-01-01'))
customers['acquisition_month'] = customers['signup_date'].dt.to_period('M')

# 파생: purchase 이벤트에서 주문 정보 추출
orders = events[events['event_type'] == 'purchase'].copy()
orders = orders.rename(columns={'event_date': 'order_time', 'order_value': 'net_amount'})

print(f'고객 수: {len(customers):,}')
print(f'이벤트 수: {len(events):,}')
print(f'주문(purchase) 수: {len(orders):,}')
print(f'\n컬럼 확인:')
print(f'  customers: {customers.columns.tolist()}')
print(f'  events   : {events.columns.tolist()}')

## 2. 이탈률 15~25% 검증 (필수 태스크 1.3)

In [ ]:
# ── 이탈률 검증 ──────────────────────────────────────────────
churn_rate = customers['churned'].mean()
churn_count = customers['churned'].sum()
total_count = len(customers)

TARGET_MIN, TARGET_MAX = 0.15, 0.25
in_range = TARGET_MIN <= churn_rate <= TARGET_MAX

print('=' * 55)
print(f'  전체 이탈률: {churn_rate:.2%}  ({churn_count:,} / {total_count:,})')
print(f'  목표 범위 : [{TARGET_MIN:.0%}, {TARGET_MAX:.0%}]')
print(f'  검증 결과 : {"✅ PASS" if in_range else "❌ FAIL"}')
print('=' * 55)

# 페르소나별 이탈률
churn_by_persona = customers.groupby('persona').agg(
    고객수=('customer_id', 'count'),
    이탈수=('churned', 'sum'),
    이탈률=('churned', 'mean'),
).sort_values('이탈률', ascending=False)

print('\n페르소나별 이탈률:')
display(churn_by_persona)

# Treatment vs Control 이탈률
tc_churn = customers.groupby('is_treatment')['churned'].agg(['count', 'sum', 'mean'])
tc_churn.columns = ['고객수', '이탈수', '이탈률']
tc_churn.index = ['Control', 'Treatment']
print('\nTreatment/Control 이탈률:')
display(tc_churn)

In [ ]:
# ── 이탈률 시각화 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (1) 전체 이탈 비율 파이
labels = [f'유지 ({total_count - churn_count:,})', f'이탈 ({churn_count:,})']
axes[0].pie([total_count - churn_count, churn_count],
            labels=labels, autopct='%1.1f%%',
            colors=['#66b3ff', '#ff6b6b'], startangle=90)
axes[0].set_title(f'전체 이탈률: {churn_rate:.1%}')

# (2) 페르소나별 이탈률 바
colors_p = sns.color_palette('Set2', len(churn_by_persona))
axes[1].barh(churn_by_persona.index, churn_by_persona['이탈률'], color=colors_p)
axes[1].axvline(TARGET_MIN, color='green', linestyle='--', alpha=0.7, label=f'목표 하한 {TARGET_MIN:.0%}')
axes[1].axvline(TARGET_MAX, color='red', linestyle='--', alpha=0.7, label=f'목표 상한 {TARGET_MAX:.0%}')
axes[1].set_title('페르소나별 이탈률')
axes[1].set_xlabel('이탈률')
axes[1].legend(fontsize=8)
for i, v in enumerate(churn_by_persona['이탈률']):
    axes[1].text(v + 0.005, i, f'{v:.1%}', va='center', fontsize=9)

# (3) Treatment vs Control
tc_labels = tc_churn.index.tolist()
tc_rates = tc_churn['이탈률'].values
axes[2].bar(tc_labels, tc_rates, color=['#ff9999', '#66b3ff'])
axes[2].set_title('Treatment vs Control 이탈률')
axes[2].set_ylabel('이탈률')
for i, v in enumerate(tc_rates):
    axes[2].text(i, v + 0.005, f'{v:.2%}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 3. 고객 기본 현황

In [ ]:
# 고객 테이블 구조
print('=== customers.csv ===')
print(f'Shape: {customers.shape}')
print(f'\nDtypes:\n{customers.dtypes}')
customers.head()

In [ ]:
# 페르소나 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

persona_counts = customers['persona'].value_counts()
colors = sns.color_palette('Set2', len(persona_counts))
axes[0].barh(persona_counts.index, persona_counts.values, color=colors)
axes[0].set_title('페르소나별 고객 수')
axes[0].set_xlabel('고객 수')
for i, v in enumerate(persona_counts.values):
    axes[0].text(v + 50, i, f'{v:,} ({v/len(customers)*100:.1f}%)', va='center')

# Treatment/Control 분포
tc_counts = customers['is_treatment'].value_counts().sort_index()
tc_labels = ['Control', 'Treatment']
axes[1].pie(tc_counts.values, labels=tc_labels, autopct='%1.1f%%',
            colors=['#66b3ff', '#ff9999'], startangle=90)
axes[1].set_title('Treatment / Control 그룹 분포')

plt.tight_layout()
plt.show()

In [ ]:
# 가입월 분포 (첫 이벤트 기준)
signup_monthly = customers['acquisition_month'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
signup_monthly.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('월별 신규 가입(첫 이벤트) 고객 수')
ax.set_xlabel('가입월')
ax.set_ylabel('고객 수')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 4. 이벤트 분석

In [ ]:
# 이벤트 유형 분포
print('=== events.csv ===')
print(f'Shape: {events.shape}')
print(f'기간: {events["event_date"].min()} ~ {events["event_date"].max()}')
print(f'\n이벤트 유형:\n{events["event_type"].value_counts()}')

fig, ax = plt.subplots(figsize=(10, 5))
event_counts = events['event_type'].value_counts()
ax.barh(event_counts.index, event_counts.values,
        color=sns.color_palette('viridis', len(event_counts)))
ax.set_title('이벤트 유형별 발생 빈도')
ax.set_xlabel('이벤트 수')
for i, v in enumerate(event_counts.values):
    ax.text(v + 1000, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 일별 이벤트 추이
daily_events = events.groupby(events['event_date'].dt.date).size()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_events.index, daily_events.values, linewidth=0.7, alpha=0.8, color='steelblue')
ax.fill_between(daily_events.index, daily_events.values, alpha=0.15, color='steelblue')
ax.set_title('일별 이벤트 발생 추이')
ax.set_xlabel('날짜')
ax.set_ylabel('이벤트 수')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
# 페르소나별 이벤트 유형 비율
# events에 이미 persona 컬럼이 있으므로 별도 merge 불필요
persona_event = pd.crosstab(events['persona'], events['event_type'], normalize='index')

fig, ax = plt.subplots(figsize=(12, 6))
persona_event.plot(kind='bar', stacked=True, ax=ax, colormap='Set3')
ax.set_title('페르소나별 이벤트 유형 구성 비율')
ax.set_xlabel('페르소나')
ax.set_ylabel('비율')
ax.legend(title='이벤트 유형', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 5. 주문(구매) 분석

In [ ]:
# 주문 기본 통계 (purchase 이벤트 기반)
print(f'=== 주문(purchase 이벤트) ===')
print(f'총 주문 건수: {len(orders):,}')
print(f'기간: {orders["order_time"].min()} ~ {orders["order_time"].max()}')
print(f'\n주문 금액(order_value) 통계:')
print(orders['net_amount'].describe().apply(lambda x: f'{x:,.0f}'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 주문 금액 분포
axes[0].hist(orders['net_amount'], bins=50, color='coral', alpha=0.7, edgecolor='white')
axes[0].set_title('주문 금액 분포')
axes[0].set_xlabel('주문 금액 (원)')
axes[0].set_ylabel('빈도')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))

# 페르소나별 평균 주문 금액 (events에 persona 포함)
avg_order = orders.groupby('persona')['net_amount'].mean().sort_values(ascending=True)
avg_order.plot(kind='barh', ax=axes[1], color=sns.color_palette('Set2', len(avg_order)))
axes[1].set_title('페르소나별 평균 주문 금액')
axes[1].set_xlabel('평균 주문 금액 (원)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))

plt.tight_layout()
plt.show()

In [ ]:
# 일별 주문 건수 및 매출 추이
daily_orders = orders.groupby(orders['order_time'].dt.date).agg(
    order_count=('customer_id', 'count'),
    total_revenue=('net_amount', 'sum')
)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(daily_orders.index, daily_orders['order_count'],
        alpha=0.5, color='steelblue', label='주문 건수')
ax1.set_ylabel('주문 건수', color='steelblue')
ax1.set_xlabel('날짜')

ax2 = ax1.twinx()
ax2.plot(daily_orders.index, daily_orders['total_revenue'],
         color='coral', linewidth=1.2, label='일 매출')
ax2.set_ylabel('일 매출 (원)', color='coral')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1e6)}M'))

ax1.set_title('일별 주문 건수 및 매출 추이')
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.show()

## 6. 이탈 고객 행동 패턴 분석

In [ ]:
# 이탈 vs 유지 고객별 기본 행동 통계
customer_stats = events.groupby('customer_id').agg(
    total_events=('event_type', 'count'),
    unique_event_types=('event_type', 'nunique'),
    active_days=('event_date', lambda x: x.dt.date.nunique()),
    first_event=('event_date', 'min'),
    last_event=('event_date', 'max'),
    purchase_count=('event_type', lambda x: (x == 'purchase').sum()),
    total_spend=('order_value', 'sum'),
)
customer_stats['active_span_days'] = (customer_stats['last_event'] - customer_stats['first_event']).dt.days

# left merge: 이벤트 없는 고객도 포함 (survivorship bias 방지)
customer_stats = customers[['customer_id', 'churned', 'persona']].merge(
    customer_stats, on='customer_id', how='left'
)
for col in ['total_events', 'unique_event_types', 'active_days', 'purchase_count', 'total_spend', 'active_span_days']:
    customer_stats[col] = customer_stats[col].fillna(0)

# 이탈/유지 비교
churn_comparison = customer_stats.groupby('churned').agg(
    고객수=('customer_id', 'count'),
    평균_이벤트수=('total_events', 'mean'),
    평균_활동일수=('active_days', 'mean'),
    평균_구매횟수=('purchase_count', 'mean'),
    평균_총지출=('total_spend', 'mean'),
    평균_활동기간=('active_span_days', 'mean'),
).round(1)
churn_comparison.index = ['유지', '이탈']

print('=== 이탈 vs 유지 고객 행동 비교 ===')
display(churn_comparison)

In [ ]:
# 이탈/유지 고객 행동 분포 비교
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('total_events', '총 이벤트 수'),
    ('active_days', '활동 일수'),
    ('purchase_count', '구매 횟수'),
    ('active_span_days', '활동 기간 (일)'),
]

for ax, (col, title) in zip(axes.flat, metrics):
    for label, churned_val in [('유지', 0), ('이탈', 1)]:
        data = customer_stats[customer_stats['churned'] == churned_val][col]
        ax.hist(data, bins=40, alpha=0.5, label=label, density=True)
    ax.set_title(title)
    ax.set_xlabel(title)
    ax.set_ylabel('밀도')
    ax.legend()

plt.suptitle('이탈 vs 유지 고객 행동 분포 비교', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. 페르소나별 이탈 심층 분석

In [ ]:
# 페르소나 × 이탈 여부별 행동 비교
persona_churn_stats = customer_stats.groupby(['persona', 'churned']).agg(
    고객수=('customer_id', 'count'),
    평균_이벤트수=('total_events', 'mean'),
    평균_구매횟수=('purchase_count', 'mean'),
    평균_총지출=('total_spend', 'mean'),
).round(1)

display(persona_churn_stats)

# 페르소나별 이탈률 바 차트
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 이탈 고객의 평균 구매 횟수 vs 유지 고객
pivot = customer_stats.pivot_table(index='persona', columns='churned',
                                    values='purchase_count', aggfunc='mean')
pivot.columns = ['유지', '이탈']
pivot.plot(kind='bar', ax=axes[0], color=['#66b3ff', '#ff6b6b'])
axes[0].set_title('페르소나별 평균 구매 횟수 (이탈 vs 유지)')
axes[0].set_ylabel('평균 구매 횟수')
axes[0].tick_params(axis='x', rotation=30)

# 이탈 고객의 평균 활동 기간
pivot2 = customer_stats.pivot_table(index='persona', columns='churned',
                                     values='active_span_days', aggfunc='mean')
pivot2.columns = ['유지', '이탈']
pivot2.plot(kind='bar', ax=axes[1], color=['#66b3ff', '#ff6b6b'])
axes[1].set_title('페르소나별 평균 활동 기간 (이탈 vs 유지)')
axes[1].set_ylabel('평균 활동 기간 (일)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 8. RFM 기초 분석 (피처 엔지니어링 준비)

In [ ]:
# 간단한 RFM 산출 (분석 기준일 = 시뮬레이션 마지막 날 + 1)
analysis_date = events['event_date'].max() + pd.Timedelta(days=1)
print(f'분석 기준일: {analysis_date.date()}')

# Recency: 마지막 구매일로부터의 경과 일수
last_purchase = orders.groupby('customer_id')['order_time'].max().rename('last_purchase_date')

# Frequency: 구매 횟수
purchase_freq = orders.groupby('customer_id').size().rename('frequency')

# Monetary: 총 지출
monetary = orders.groupby('customer_id')['net_amount'].sum().rename('monetary')

# 모든 고객 포함 (비구매자도 포함하여 편향 방지)
rfm = customers[['customer_id', 'churned', 'persona', 'signup_date']].copy()
rfm = rfm.merge(last_purchase, on='customer_id', how='left')
rfm = rfm.merge(purchase_freq, on='customer_id', how='left')
rfm = rfm.merge(monetary, on='customer_id', how='left')
rfm[['frequency', 'monetary']] = rfm[['frequency', 'monetary']].fillna(0)
rfm['recency_days'] = np.where(
    rfm['last_purchase_date'].notna(),
    (analysis_date - rfm['last_purchase_date']).dt.days,
    (analysis_date - rfm['signup_date']).dt.days
)
rfm = rfm.drop(columns=['signup_date'])

print(f'\nRFM 산출 고객 수: {len(rfm):,} (비구매자 포함 전체 고객)')
print(f'  구매 이력 있는 고객: {(rfm["frequency"] > 0).sum():,}')
print(f'  구매 이력 없는 고객: {(rfm["frequency"] == 0).sum():,}')
print(f'\nRFM 기초 통계:')
display(rfm[['recency_days', 'frequency', 'monetary']].describe().round(1))

In [ ]:
# RFM 분포 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title, color in [
    (axes[0], 'recency_days', 'Recency (마지막 구매 후 경과일)', 'steelblue'),
    (axes[1], 'frequency', 'Frequency (구매 횟수)', 'coral'),
    (axes[2], 'monetary', 'Monetary (총 지출액)', 'seagreen'),
]:
    for label, churned_val in [('유지', 0), ('이탈', 1)]:
        data = rfm[rfm['churned'] == churned_val][col]
        ax.hist(data, bins=40, alpha=0.5, label=label, density=True)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.legend()

plt.suptitle('RFM 분포 - 이탈 vs 유지', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 9. 코호트 리텐션 개요

In [ ]:
# 간단한 코호트 리텐션 (가입월 기준, 월별 활동 여부)
events_monthly = events.copy()
events_monthly['event_month'] = events_monthly['event_date'].dt.to_period('M')
events_monthly = events_monthly.merge(
    customers[['customer_id', 'acquisition_month']], on='customer_id'
)

# 코호트 period 계산
events_monthly['cohort_num'] = events_monthly['acquisition_month'].apply(lambda x: x.ordinal)
events_monthly['event_month_num'] = events_monthly['event_month'].apply(lambda x: x.ordinal)
events_monthly['period'] = events_monthly['event_month_num'] - events_monthly['cohort_num']

# 코호트 사이즈
cohort_sizes = customers.groupby('acquisition_month')['customer_id'].nunique()

# 기간별 활성 고객 수
cohort_active = (
    events_monthly[events_monthly['period'] >= 0]
    .groupby(['acquisition_month', 'period'])['customer_id']
    .nunique()
    .reset_index(name='active_customers')
)
cohort_active['cohort_size'] = cohort_active['acquisition_month'].map(cohort_sizes)
cohort_active['retention_rate'] = cohort_active['active_customers'] / cohort_active['cohort_size']

# 히트맵
pivot = cohort_active.pivot_table(
    index='acquisition_month', columns='period', values='retention_rate'
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd_r', ax=ax,
            vmin=0, vmax=1, linewidths=0.5)
ax.set_title('코호트별 리텐션율 히트맵 (가입월 기준, 전체 이벤트 활동)')
ax.set_xlabel('기간 (개월)')
ax.set_ylabel('가입 코호트')
plt.tight_layout()
plt.show()

# 마일스톤 요약
milestones = [1, 3, 6]
for m in milestones:
    if m in pivot.columns:
        avg_ret = pivot[m].mean()
        print(f'M{m} 평균 리텐션: {avg_ret:.2%}')

## 10. 주요 피처 상관관계 (탐색)

In [ ]:
# 고객 수준 피처 통합 + 상관관계
analysis_df = customer_stats[['customer_id', 'total_events', 'active_days',
                               'purchase_count', 'total_spend', 'active_span_days',
                               'churned']].copy()

# RFM 합치기
analysis_df = analysis_df.merge(
    rfm[['customer_id', 'recency_days', 'frequency', 'monetary']],
    on='customer_id', how='left'
)

numeric_cols = ['total_events', 'active_days', 'purchase_count', 'total_spend',
                'active_span_days', 'recency_days', 'frequency', 'monetary', 'churned']
corr = analysis_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('주요 피처 간 상관관계')
plt.tight_layout()
plt.show()

# churned와의 상관 정리
churn_corr = corr['churned'].drop('churned').sort_values()
print('\n=== churned 컬럼과의 상관 ===')
for feat, val in churn_corr.items():
    print(f'  {feat:25s}: {val:+.3f}')

## 11. EDA 요약 및 인사이트

### 주요 발견 사항
1. **이탈률 검증**: 시뮬레이터 생성 데이터의 전체 이탈률이 15~25% 목표 범위 내인지 확인 완료
2. **페르소나별 행동 차이**: 각 페르소나는 방문 빈도, 구매 전환율, 이탈률에서 뚜렷한 차이
3. **이탈 고객 행동 특성**: 이탈 고객은 활동 기간, 구매 횟수, 총 지출에서 유지 고객 대비 낮은 수치
4. **RFM 분포**: Recency가 높고 Frequency/Monetary가 낮은 고객이 이탈 경향
5. **Treatment 효과**: Treatment vs Control 간 이탈률 차이를 초기 확인

### 후속 분석 방향
- `src/analysis/cohort.py`: M1, M3, M6 코호트 리텐션 곡선 상세 분석
- 이탈 고객의 마지막 30일 행동 시퀀스 패턴 분석
- 고객 여정 퍼널(가입 → 첫구매 → 재구매 → 충성 → 이탈) 전환율 분석
- RFM + 행동 변화율 피처 본격 엔지니어링